> Mostly follows priyam's lecture

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


In [4]:
transform = transforms.Compose(
    [
        transforms.Resize((32,32)), 
        transforms.ToTensor()
    ]
)

train_set = MNIST("../data/mnist/", train=True, transform=transform)
test_set = MNIST("../data/mnist/", train=False, transform=transform)

device = "cuda" if torch.cuda.is_available() else "cpu"

### Conv auto-encoder

In [6]:
for img, label in train_set:
    print(img, label)
    break

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]]) 5


In [24]:
batch_size = 4
train = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val = DataLoader(test_set, batch_size=batch_size, shuffle=False)

for x, y in train:
    print(x.shape, y.shape)
    break

torch.Size([4, 1, 32, 32]) torch.Size([4])


In [45]:
class ConvAutoEncoder(nn.Module):
    def __init__(self, in_channels = 1, bottleneck = 4):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d( in_channels=in_channels, out_channels= 8, kernel_size=3, stride=2, padding=1, bias=False ),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d( in_channels=8, out_channels= 16, kernel_size=3, padding=1, stride=2, bias=False ),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d( in_channels=16, out_channels= bottleneck, kernel_size=3, padding=1, stride=2),

        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d( in_channels = bottleneck, out_channels=16, kernel_size=3, padding=1, stride=2, output_padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d( in_channels = 16, out_channels=8, kernel_size=3, padding=1, stride=2, output_padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.ConvTranspose2d( in_channels = 8, out_channels=in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward_enc(self, x):
        return self.encoder(x)

    def forward_dec(self, x):
        return self.decoder(x)
    
    def forward(self, x):
        enc = self.encoder(x)
        dec = self.decoder(enc)
        return enc, dec


model = ConvAutoEncoder( ) 
for x, y in train:
    enc, dec = model(x)
    print(x.shape, enc.shape, dec.shape)
    break
model

torch.Size([4, 1, 32, 32]) torch.Size([4, 4, 4, 4]) torch.Size([4, 1, 32, 32])


ConvAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(16, 4, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
  (decoder): Sequential(
    (0): ConvTranspose2d(4, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): ConvTranspose2d(16, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
    (4): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): ConvTranspose2d(8, 1, kernel_size=(3, 3), str